In [1]:
# =============================================================================
# MODEL 2 - RANDOM FOREST REGRESSOR
# Task: Predict Crime_Count based on Location and Date
# =============================================================================

import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================
DATA_PATH = "./crime_per_capita_df.csv"
TARGET = "Crime_Count"

TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEAR = 2024

BASELINE_PARAMS = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
    "n_jobs": -1,
}

TUNED_CONFIGS = [
    {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 4},
    {"n_estimators": 400, "max_depth": 30, "min_samples_leaf": 6},
]

NO_LAG_PARAMS = {
    "n_estimators": 200,
    "max_depth": 20,
    "min_samples_leaf": 2,
    "random_state": 42,
    "n_jobs": -1,
}

HOTSPOT_PERCENTS = [5, 10, 20]
REPORT_TXT_PATH = "evaluation_results_model_02.txt"
MODEL_COMPARISON_PATH = "model_comparison.csv"
ARTIFACT_PATH = Path("artifacts") / "best_model_model_02.joblib"

FEATURES = [
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",
    "Year",
    "Month",
    "month_sin",
    "month_cos",
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
]

# =============================================================================
# 2. HELPERS
# =============================================================================
def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator == 0.0:
        return np.nan
    return float(numerator) / denominator


def round_or_nan(value, digits=4):
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), digits)


def fmt4(value):
    if pd.isna(value) or not np.isfinite(value):
        return "nan"
    return f"{float(value):.4f}"


def evaluate(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'=' * 50}")
    print(f"  {model_name}")
    print(f"{'=' * 50}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R2    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "MAPE": round(mape, 2),
    }


# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI — Prediction Accuracy Index
# RRI — Recapture Rate Index
# PEI — Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10, model_name="Model"):
    if not (0 < hotspot_percent <= 100):
        raise ValueError("hotspot_percent must be in (0, 100].")

    df_eval = pd.DataFrame(
        {
            "actual": np.asarray(y_true, dtype=float).reshape(-1),
            "predicted": np.asarray(y_pred, dtype=float).reshape(-1),
        }
    ).dropna().reset_index(drop=True)

    if df_eval.empty:
        return {
            "Model": model_name,
            "Hotspot_%": hotspot_percent,
            "Crimes_Captured": 0,
            "Total_Crimes": 0,
            "RRI": np.nan,
            "PAI": np.nan,
            "PEI": np.nan,
        }

    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    crimes_in_hotspots = float(df_eval.loc[df_eval["hotspot"], "actual"].sum())
    total_crimes = float(df_eval["actual"].sum())
    hotspot_area = int(df_eval["hotspot"].sum())
    total_area = int(len(df_eval))

    rri = safe_divide(crimes_in_hotspots, total_crimes)
    area_ratio = safe_divide(hotspot_area, total_area)
    pai = safe_divide(rri, area_ratio) if pd.notna(rri) and pd.notna(area_ratio) else np.nan
    pei = pai  # simplified PEI approximation

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Model             : {model_name}")
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {fmt4(rri)}")
    print(f"PAI               : {fmt4(pai)}")
    print(f"PEI               : {fmt4(pei)}")

    return {
        "Model": model_name,
        "Hotspot_%": hotspot_percent,
        "Crimes_Captured": int(round(crimes_in_hotspots)),
        "Total_Crimes": int(round(total_crimes)),
        "RRI": round_or_nan(rri, 4),
        "PAI": round_or_nan(pai, 4),
        "PEI": round_or_nan(pei, 4),
    }


def write_evaluation_report(report_path, model_results_df, hotspot_df, best_row, artifact_path):
    reg_cols = ["Model", "MAE", "RMSE", "R2", "MAPE"]
    hs_cols = ["Model", "Hotspot_%", "Crimes_Captured", "Total_Crimes", "RRI", "PAI", "PEI"]

    reg_out = model_results_df[reg_cols].copy().sort_values("MAE")
    hs_out = hotspot_df[hs_cols].copy().sort_values(["Model", "Hotspot_%"])

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# =============================================================================\n")
        f.write("# CRIME HOTSPOT EVALUATION METRICS\n")
        f.write("# PAI — Prediction Accuracy Index\n")
        f.write("# RRI — Recapture Rate Index\n")
        f.write("# PEI — Prediction Efficiency Index\n")
        f.write("# =============================================================================\n\n")
        f.write("Definitions\n")
        f.write("RRI = Crimes captured in hotspots / Total actual crimes\n")
        f.write("PAI = RRI / Hotspot area ratio\n")
        f.write("PEI = RRI / Hotspot area ratio (simplified approximation)\n\n")
        f.write("# =============================================================================\n")
        f.write("# REGRESSION EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(reg_out.to_string(index=False))
        f.write("\n\n")
        f.write("# =============================================================================\n")
        f.write("# HOTSPOT EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(hs_out.to_string(index=False))
        f.write("\n\n")
        f.write("# =============================================================================\n")
        f.write("# BEST MODEL EXPORT SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(f"Model selected : {best_row['Model']}\n")
        f.write("Metric used    : MAE\n")
        f.write(f"MAE            : {best_row['MAE']}\n")
        f.write(f"RMSE           : {best_row['RMSE']}\n")
        f.write(f"R2             : {best_row['R2']}\n")
        f.write(f"MAPE           : {best_row['MAPE']}\n")
        f.write(f"Artifact path  : {artifact_path}\n")


# =============================================================================
# 3. LOAD DATA
# =============================================================================
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs: {df['LSOA_Code'].nunique()}")

# =============================================================================
# 4. FEATURE VALIDATION + TRAIN/TEST SPLIT
# =============================================================================
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING - missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

print(f"\nUsing {len(FEATURES)} features -> target: '{TARGET}'")

train = df[df["Year"].isin(TRAIN_YEARS)].copy()
test = df[df["Year"] == TEST_YEAR].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print(f"Train : {len(X_train):,} rows  |  Test : {len(X_test):,} rows")

# =============================================================================
# 5. BASELINE + TUNED RANDOM FOREST RUNS
# =============================================================================
results = []

print("\n--- Fitting Baseline Random Forest ---")
rf_baseline = RandomForestRegressor(**BASELINE_PARAMS)
rf_baseline.fit(X_train, y_train)
preds_baseline = np.clip(rf_baseline.predict(X_test), 0, None)
res_baseline = evaluate("RF - baseline (100 trees, no depth limit)", y_test, preds_baseline)

results.append(
    {
        **res_baseline,
        "model_obj": rf_baseline,
        "preds": preds_baseline,
        "feature_set": FEATURES.copy(),
    }
)

print("\n--- Fitting Tuned Random Forest Variants ---")
for cfg in TUNED_CONFIGS:
    label = (
        f"RF - {cfg['n_estimators']} trees, "
        f"depth={cfg['max_depth']}, leaf={cfg['min_samples_leaf']}"
    )
    print(f"\n--- Fitting {label} ---")

    rf = RandomForestRegressor(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        min_samples_leaf=cfg["min_samples_leaf"],
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    preds = np.clip(rf.predict(X_test), 0, None)
    res = evaluate(label, y_test, preds)

    results.append(
        {
            **res,
            "model_obj": rf,
            "preds": preds,
            "feature_set": FEATURES.copy(),
        }
    )

# =============================================================================
# 6. NO-LAG VARIANT
# =============================================================================
print("\n--- Fitting RF WITHOUT lag features (honest spatial model) ---")

lag_cols = [f for f in FEATURES if "lag" in f or "rolling" in f or "msoa_avg" in f]
no_lag_features = [f for f in FEATURES if f not in lag_cols]

print(f"Features used (no lags): {len(no_lag_features)}")

rf_no_lag = RandomForestRegressor(**NO_LAG_PARAMS)
rf_no_lag.fit(X_train[no_lag_features], y_train)
preds_no_lag = np.clip(rf_no_lag.predict(X_test[no_lag_features]), 0, None)
res_no_lag = evaluate("RF - no lag features (spatial only)", y_test, preds_no_lag)

results.append(
    {
        **res_no_lag,
        "model_obj": rf_no_lag,
        "preds": preds_no_lag,
        "feature_set": no_lag_features,
    }
)

# =============================================================================
# 7. PICK BEST MODEL
# =============================================================================
best = min(results, key=lambda x: x["MAE"])
best_rf = best["model_obj"]
best_features = best["feature_set"]
best_preds = best["preds"]

print(f"\nBest configuration: {best['Model']}")
print(f"MAE={best['MAE']}  RMSE={best['RMSE']}  R2={best['R2']}  MAPE={best['MAPE']}%")
print(f"Features in best model: {len(best_features)}")

# =============================================================================
# 8. FEATURE IMPORTANCE
# =============================================================================
print("\n--- Feature Importances (Top 15) ---")
fi = pd.DataFrame(
    {
        "Feature": best_features,
        "Importance": best_rf.feature_importances_,
    }
).sort_values("Importance", ascending=False)

print(fi.head(15).to_string(index=False))

lag_imp = fi[fi["Feature"].isin(lag_cols)]["Importance"].sum()
other_imp = 1 - lag_imp
print(f"\nLag/rolling features combined importance : {lag_imp * 100:.1f}%")
print(f"All other features combined importance   : {other_imp * 100:.1f}%")

# =============================================================================
# 9. PREDICTION INTERVALS
# =============================================================================
print("\n--- Prediction Intervals (uncertainty estimate) ---")
X_test_best = X_test[best_features]

all_tree_preds = np.array([tree.predict(X_test_best) for tree in best_rf.estimators_])
pred_mean = all_tree_preds.mean(axis=0)
pred_std = all_tree_preds.std(axis=0)
pred_low = np.clip(pred_mean - 1.96 * pred_std, 0, None)
pred_high = pred_mean + 1.96 * pred_std

sample_idx = test.head(8).index
sample_df = test.loc[sample_idx, ["LSOA_Code", "Year", "Month", TARGET]].copy()
sample_df["Predicted"] = np.round(pred_mean[:8], 1)
sample_df["Lower_95"] = np.round(pred_low[:8], 1)
sample_df["Upper_95"] = np.round(pred_high[:8], 1)
sample_df["Actual"] = y_test.values[:8]
print(sample_df.to_string(index=False))

# =============================================================================
# 10. ERROR ANALYSIS
# =============================================================================
print("\n--- Error Analysis ---")
error_df = test[["LSOA_Code", "Year", "Month", TARGET]].copy()
error_df["Predicted"] = np.round(pred_mean, 1)
error_df["Error"] = np.abs(error_df[TARGET] - error_df["Predicted"])
error_df["Pct_Error"] = (error_df["Error"] / (error_df[TARGET] + 1e-9)) * 100

worst_lsoas = error_df.groupby("LSOA_Code")["Error"].mean().sort_values(ascending=False).head(10)
print("\nTop 10 LSOAs with highest average prediction error:")
print(worst_lsoas.to_string())

monthly_err = (
    error_df.groupby("Month")["Error"]
    .mean()
    .reset_index()
    .rename(columns={"Error": "Avg_MAE"})
)
print("\nAverage error by month (1=Jan ... 12=Dec):")
print(monthly_err.to_string(index=False))

# =============================================================================
# 11. APPEND TO MODEL COMPARISON CSV
# =============================================================================
new_rows = pd.DataFrame(
    [
        {k: v for k, v in r.items() if k not in ["model_obj", "preds", "feature_set"]}
        for r in results
    ]
)

try:
    existing = pd.read_csv(MODEL_COMPARISON_PATH)
    combined = pd.concat([existing, new_rows], ignore_index=True)
except FileNotFoundError:
    combined = new_rows

combined.to_csv(MODEL_COMPARISON_PATH, index=False)
print("\nUpdated model_comparison.csv:")
print(combined[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))

# =============================================================================
# 12. CRIME HOTSPOT EVALUATION
# =============================================================================
print("\n--- Crime Forecasting Indices ---")
hotspot_results = []

for pct in HOTSPOT_PERCENTS:
    res = crime_hotspot_metrics(
        y_true=y_test.values,
        y_pred=pred_mean,
        hotspot_percent=pct,
        model_name=best["Model"],
    )
    hotspot_results.append(res)

hotspot_df = pd.DataFrame(hotspot_results)

print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))

# =============================================================================
# 13. EXPORT BEST MODEL
# =============================================================================
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

best_model_package = {
    "model_name": best["Model"],
    "model": best_rf,
    "features_used": best_features,
    "all_candidate_features": FEATURES,
    "target": TARGET,
    "metric_used": "MAE",
    "metrics": {
        "MAE": best["MAE"],
        "RMSE": best["RMSE"],
        "R2": best["R2"],
        "MAPE": best["MAPE"],
    },
    "train_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "hotspot_percents": HOTSPOT_PERCENTS,
    "is_no_lag_model": set(best_features) != set(FEATURES),
}

joblib.dump(best_model_package, ARTIFACT_PATH)

print("\nBest model export complete")
print(f"Model selected : {best['Model']}")
print(f"Saved artifact : {ARTIFACT_PATH}")

# =============================================================================
# 14. WRITE SINGLE TEXT REPORT (OVERWRITE EACH RUN)
# =============================================================================
model_results_df = pd.DataFrame(
    [
        {k: v for k, v in r.items() if k not in ["model_obj", "preds", "feature_set"]}
        for r in results
    ]
)

write_evaluation_report(
    report_path=REPORT_TXT_PATH,
    model_results_df=model_results_df,
    hotspot_df=hotspot_df,
    best_row=best,
    artifact_path=ARTIFACT_PATH,
)

print(f"\nEvaluation report saved to {REPORT_TXT_PATH}")

Dataset shape: (304336, 39)
Date range: 2020 to 2024
Unique LSOAs: 12415

Using 30 features -> target: 'Crime_Count'
Train : 241,710 rows  |  Test : 62,626 rows

--- Fitting Baseline Random Forest ---

  RF - baseline (100 trees, no depth limit)
  MAE   : 4.4442
  RMSE  : 8.6097
  R2    : 0.9431
  MAPE  : 41.06%

--- Fitting Tuned Random Forest Variants ---

--- Fitting RF - 200 trees, depth=10, leaf=2 ---

  RF - 200 trees, depth=10, leaf=2
  MAE   : 4.3752
  RMSE  : 8.1562
  R2    : 0.9489
  MAPE  : 40.32%

--- Fitting RF - 300 trees, depth=20, leaf=4 ---

  RF - 300 trees, depth=20, leaf=4
  MAE   : 4.3883
  RMSE  : 8.4391
  R2    : 0.9453
  MAPE  : 40.00%

--- Fitting RF - 400 trees, depth=30, leaf=6 ---

  RF - 400 trees, depth=30, leaf=6
  MAE   : 4.3997
  RMSE  : 8.6254
  R2    : 0.9429
  MAPE  : 40.16%

--- Fitting RF WITHOUT lag features (honest spatial model) ---
Features used (no lags): 21

  RF - no lag features (spatial only)
  MAE   : 6.4724
  RMSE  : 16.9465
  R2    : 0.